In [15]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict  
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from pydantic import BaseModel, Field
from typing import Annotated
import operator
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage, AIMessage
from langgraph.graph.message import add_messages

from langgraph.checkpoint.memory import MemorySaver



In [16]:
load_dotenv()
model = ChatGoogleGenerativeAI(
    model = "gemini-3.5-flash-lite"
)

In [17]:
class chatState(TypedDict):
    messages : Annotated[list[BaseMessage],add_messages]

In [18]:
def chat_node(state : chatState):
    messages = state['messages']

    response = model.invoke(messages)

    return {'messages' : [response]}

In [19]:
graph = StateGraph(chatState)
checkpointer = MemorySaver()

graph.add_node('chat_node', chat_node)

graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)

chatbot = graph.compile(checkpointer=checkpointer)

In [14]:
#in this  the chatbot does not have a memory as everytime
#we invoke the chatbot the state is getting created from the scratch

#we use persistance to overcome this in  next cell

while True:
    user_message = input("Type here: ").strip()
    print("User:", user_message)

    if user_message.lower() in ["exit", "quit"]:
        break

    if not user_message:
        continue

    response = chatbot.invoke({
        "messages": [HumanMessage(content=user_message)]
    })

    ai_content = response["messages"][-1].content

    if isinstance(ai_content, str):
        print("AI:", ai_content)

    elif isinstance(ai_content, list):
        for block in ai_content:
            if isinstance(block, dict) and block.get("type") == "text":
                print("AI:", block["text"])


User: quit


In [20]:
thread_id = '1'

while True:
    user_message = input("Type here: ").strip()
    print("User:", user_message)

    if user_message.lower() in ["exit", "quit"]:
        break

    if not user_message:
        continue

    config = {'configurable' : {'thread_id' : thread_id}}   

    response = chatbot.invoke({
        "messages": [HumanMessage(content=user_message)]},
        config= config
        )

    ai_content = response["messages"][-1].content

    if isinstance(ai_content, str):
        print("AI:", ai_content)

    elif isinstance(ai_content, list):
        for block in ai_content:
            if isinstance(block, dict) and block.get("type") == "text":
                print("AI:", block["text"])


User: hello my name is krishna
AI: Hello Krishna! It's nice to meet you. How can I help you today?
User: what is my name
AI: Your name is Krishna.
User: exit
